In [ ]:
# PACKAGES
#-----------------------------------------------------
import pyvisa as visa
from tm_data_types import AnalogWaveform, write_file
from tekhsi import TekHSIConnect
import pylablib as pll
from pylablib.devices import Thorlabs
from pymeasure.instruments.teledyne import teledyneT3AFG
from datetime import datetime
import matplotlib.pyplot as plot
import numpy as np
import struct
import math
import os
from tqdm.notebook import tqdm
from IPython.display import clear_output
#---------------------------------------------------
import pandas as pd
import time
import argparse
import csv
import sys
from collections import defaultdict
from pathlib import Path
from scipy.spatial import cKDTree
from matplotlib.animation import FuncAnimation
from matplotlib.colors import LinearSegmentedColormap
from shapely.geometry import LineString, Point
from scipy.spatial.distance import cdist

In [81]:
# =============================================================================
# DEFAULT PATHS  (Windows -- adjust for your setup)

pos_file  = r'C:\Users\tvreugdenhil\Desktop\juan\Figure8data\pos_grid_full.csv'
disp_file = r'C:\Users\tvreugdenhil\Desktop\juan\Figure8data\disp_grid_full.csv'
visit_log  = r'C:\Users\tvreugdenhil\Desktop\juan\Image\figure1\visit_log_f1_rotated_0.csv'
scan_zone  = r'C:\Users\tvreugdenhil\Desktop\juan\Image\figure2\scan_zone_f1_rotated_0.csv'
output_dir = r'C:\Users\tvreugdenhil\Desktop\juan\Image\figure4'
#Temporal sampling step (s) of the simulated/acquired displacement signals.

DEFAULT_DT = 1 / 750000 / 6

In [84]:
# CONNECT SCANNING STAGES
#-----------------------------------------------------
scan_stage  =   Thorlabs.kinesis.KinesisMotor("70366044", is_rack_system=True)
scan_stage.get_full_info()

{'velocity_parameters': TVelocityParams(min_velocity=0.0, acceleration=352234.9676841915, max_velocity=23.84500745156483),
 'jog_parameters': TJogParams(mode='step', step_size=1280, min_velocity=0.0, acceleration=0.0, max_velocity=5.961251862891207, stop_mode='profiled'),
 'homing_parameters': THomeParams(home_direction='reverse', limit_switch='reverse', velocity=0.0, offset_distance=12800),
 'gen_move_parameters': TGenMoveParams(backlash_distance=0),
 'limit_switch_parameters': TLimitSwitchParams(hw_kind_cw='make', hw_kind_ccw='make', hw_swapped=False, sw_position_cw=0, sw_position_ccw=0, sw_kind='ignore'),
 'position': 0,
 'status': ['tracking', 'digio1', 'enabled'],
 'cls': 'KinesisMotor',
 'conn': {'port': '70366044',
  'baudrate': 115200,
  'bytesize': 8,
  'parity': 'N',
  'stopbits': 1,
  'xonxoff': 0,
  'rtscts': True},
 'device_info': TDeviceInfo(serial_no=70000000, model_no='BSC203', fw_ver='2.3.4', hw_type=32, hw_ver=2, mod_state=0, nchannels=3, notes='APT Stepper Motor Cont

In [85]:
# FUNCTIONS FOR STAGE CONTROL
#-----------------------------------------------------
def stage_motion_x(step_size_x, direction_x):
    # motion parameters
    steps_per_mm    =   409600 #microsteps/mm
    min_vel         =   0.0
    max_vel         =   0.3*steps_per_mm #steps/s
    accel           =   10*steps_per_mm #steps/s*s
    actual_max_vel = max_vel / steps_per_mm #mm/s
    actual_accel = accel / steps_per_mm #mm/s*s
    # steps written to driver
    steps_x         =   steps_per_mm * step_size_x / 1000  
    scan_stage.setup_gen_move(backlash_distance=0, channel=1, scale=True)
    scan_stage.setup_velocity(min_velocity=min_vel, acceleration=accel, max_velocity=max_vel, channel=1, scale=True)
    scan_stage.setup_jog(mode='step', step_size=steps_x, min_velocity=min_vel, acceleration=accel, max_velocity=max_vel, stop_mode='profiled', channel=1, scale=True)# motion X
    print(time.ctime()  +   '   Moving stage in X- by %s microns...' % (str(step_size_x)))
    scan_stage.move_by(direction_x * steps_x, channel=1)
    # get current position
    curr_pos_x    =   scan_stage.get_position(channel=1, scale=True)
    print(time.ctime()  +   '   Current position (%s)' % (str(curr_pos_x)))

def stage_motion_y(step_size_y, direction_y):
    # motion parameters
    steps_per_mm    =   409600 #microsteps/mm
    min_vel         =   0.0
    max_vel         =   0.3*steps_per_mm #steps/s
    accel           =   10*steps_per_mm #steps/s*s
    actual_max_vel = max_vel / steps_per_mm #mm/s
    actual_accel = accel / steps_per_mm #mm/s*s
    # steps written to driver
    steps_y         =   steps_per_mm * step_size_y / 1000  
    scan_stage.setup_gen_move(backlash_distance=0, channel=2, scale=True)
    scan_stage.setup_velocity(min_velocity=min_vel, acceleration=accel, max_velocity=max_vel, channel=2, scale=True)
    scan_stage.setup_jog(mode='step', step_size=steps_y, min_velocity=min_vel, acceleration=accel, max_velocity=max_vel, stop_mode='profiled', channel=2, scale=True)# motion X
    print(time.ctime()  +   '   Moving stage in Y- by %s microns...' % (str(step_size_y)))
    scan_stage.move_by(direction_y * steps_y, channel=2)
    # get current position
    curr_pos_y    =   scan_stage.get_position(channel=2, scale=True)
    print(time.ctime()  +   '   Current position (%s)' % (str(curr_pos_y)))


In [ ]:

# =============================================================================
# 1. MODULAR FUNCTIONS FOR LOCAL TRACKING
# =============================================================================

def generate_semi_circle_points(curr_x, curr_y, cur_angle, R, nh,ang_probe_del):
    """
    Generates nh points on a semi-circle of radius R oriented along the cur_angle direction.
    """
    points = []
    angs = np.linspace(cur_angle - np.pi/2, cur_angle + np.pi/2, nh)
    for theta in angs:
        for probe in (-ang_probe_del,0.0,ang_probe_del):
            a=theta+np.deg2rad(probe)
            x=curr_x+R*np.cos(a)
            y=curr_y+R*np.sin(a)
            points.append((x,y))
    points_array=np.array(points, dtype=float)
    #pts = np.column_stack([curr_x + R * np.cos(angs), curr_y + R * np.sin(angs)])
    return points_array[1:-1]

def measure_signal_at_point(pt_x, pt_y, tree_model, disp_path, pos_df, norm_val):
    """
    Projects a candidate point onto the mesh via KDTree, reads the corresponding signal,
    from the pre-loaded Dataframe, and extracts the normalized maximum amplitude. 
    """
    _, idx_pt = tree_model.query([pt_x, pt_y])
    real_x, real_y, real_z = pos_df.iloc[idx_pt].values
    #Instantaneous lecture in the RAM
    sig = disp_path.iloc[:, idx_pt].values
    idx_t = np.abs(sig).argmax()
    amp = np.abs(sig[idx_t])
    amp_norm = amp / norm_val
    return real_x, real_y, amp_norm, idx_t

def select_best_candidate(candidates, t_old, tol=3):
    """
    Selects the best direction among the candidates.
    Criteria: t_new > t_old - tol AND Maximum amplitude.
    If no candidate meets the time criterion, applies a fallback behavior.
    """
    # Filter candidates meeting the temporal constraint
    valid_candidates = [c for c in candidates if c[3] > (t_old - tol)]
    
    if valid_candidates:
        # Select the max amplitude among valid candidates
        best = max(valid_candidates, key=lambda item: item[2])
        fallback_used = False
    else:
        # Fallback behavior: select the max amplitude among ALL candidates (ignore t)
        best = max(candidates, key=lambda item: item[2])
        fallback_used = True
        
    return best, fallback_used




In [ ]:
# Automated linescan code. You can ignore everything regarding the scope and function generator. Feel free to use the rest of the code as example.

#-----------------------------------------------------
# SAVE SETTINGS -- user input
#-----------------------------------------------------
now     =   datetime.now()
current =   now.strftime('%y/%m/%d: %H:%M')
# folder  =   input('Enter master folder name/Experiment ID: ')
folder  =   'Test10runs'
data_path       =   r'D:\AcousticMetamat_DATA' + r'\%s' % folder
data_path_scope =   r'E:\PhD\Lab\Automated\DamageModel' + r'\%s' % folder
save_chn      =   ['CH1', 'CH2']   # channels to save 
if not os.path.exists(data_path):
    os.makedirs(data_path)
flag_motion     =   0   # 0: no motion
number_points   =   10
step_x          =   10 # microns
step_x_dir      =   -1

#-----------------------------------------------------
# ARM OSCILLOSCOPE FOR DATA COLLECTION
#-----------------------------------------------------
# write settings to scope
channelsetup(scope,chsetup,n_active_channel)
timebase(scope, acquisition_time, sampling_rate, pretrigger)
trigger_setup(trig_mode=trig_mode, trig_polarity=trig_polarity,trig_source=trig_source,trig_level=trig_level)
data_acquisition(acquisition[acq], num_average, None)
# GET TRIGGER STATE BEFORE EXPERIMENT
print(time.ctime()  +   '   Scope trigger status %s...' % scope.query('trigger:state?').strip())
#-----------------------------------------------------
# ACQUIRE DATA
#-----------------------------------------------------
# ----------------------------
# Setup progress bar
# ----------------------------
pbar = tqdm(total=number_points,
            ncols=100,
            unit="acq",
            dynamic_ncols=True,
            mininterval=0.5)

# Arm function generator
fg.arm_ch2_for_manual_pulses()
# Acquisition loop
for I in range(number_points):

    timestamp = time.strftime('%H:%M:%S')  # current time

    # --- Update progress bar description ---
    desc = f"{timestamp} | Acq #{I+1}/{number_points} | Status: Configuring scope"
    pbar.set_description(desc)

    # Build file path
    data_file      = f"\\test{I}.csv"
    data_full_path = data_path + data_file

    # --- Configure scope ---
    channelsetup(scope, chsetup, n_active_channel)
    timebase(scope, acquisition_time, sampling_rate, pretrigger)
    trigger_setup(trig_mode=trig_mode,
                  trig_polarity=trig_polarity,
                  trig_source=trig_source,
                  trig_level=trig_level)
    data_acquisition(acquisition[acq], num_average, None) #setup scope for acquisition on trigger

    # --- Wait for scope to be ready ---
    while scope.query('trigger:state?').strip() != 'READY':
        time.sleep(0.1)

    # --- Trigger function generator ---
    timestamp = time.strftime('%H:%M:%S')
    desc = f"{timestamp} | Acq #{I+1}/{number_points} | Status: Triggering FG"
    pbar.set_description(desc)
    fg.trigger_ch2_pulse()

    # --- Wait for acquisition to complete ---
    timestamp = time.strftime('%H:%M:%S')
    desc = f"{timestamp} | Acq #{I+1}/{number_points} | Status: Waiting for acquisition"
    pbar.set_description(desc)
    while int(scope.query('acquire:state?')) != 0:
        time.sleep(0.1)

    # --- Save waveform data ---
    timestamp = time.strftime('%H:%M:%S')
    desc = f"{timestamp} | Acq #{I+1}/{number_points} | Status: Saving data"
    pbar.set_description(desc)
    wfm_out, wfm_raw = save_data(save_chan=save_chn, data_full_path=data_full_path) #save acquisition to lab PC
    #setup_save_on(data_path_scope) #save acquisition to scope internal storage

    timestamp = time.strftime('%H:%M:%S')
    if os.path.isfile(data_full_path) and os.access(data_full_path, os.R_OK):
        status = "Data saved"
    else:
        status = "Error saving data"
        desc = f"{timestamp} | Acq #{I+1}/{number_points} | Status: {status}"
        pbar.set_description(desc)
        pbar.close()
        raise RuntimeError(f"{timestamp}   Error saving data for acquisition #{I+1}")

        # --- Stage motion if applicable ---
    if I < (number_points - 1) and flag_motion == 1:
        timestamp = time.strftime('%H:%M:%S')
        desc = f"{timestamp} | Acq #{I+1}/{number_points} | Status: Stage moving"
        pbar.set_description(desc)
        stage_motion_x(step_size_x=step_x, direction_x=step_x_dir)
        while scan_stage.is_moving():
            time.sleep(0.1)
        time.sleep(2.5)  # small delay after motion
        
    # --- Update progress bar with final status ---
    desc = f"{timestamp} | Acq #{I+1}/{number_points} | Status: {status}"
    pbar.set_description(desc)
    pbar.update(1)

# ----------------------------
# Cleanup after loop
# ----------------------------
fg.disarm_ch2()
scope.write('LOCK NONE')

# Return stage to starting point
if flag_motion == 1:
    tqdm.write(f"{time.ctime()}   Homing stage...")
    stage_motion_x(step_size_x=step_x * (number_points - 1),
                   direction_x=step_x_dir * -1)
    while scan_stage.is_moving():
        time.sleep(0.1)
    tqdm.write(f"{time.ctime()}   Stage homing complete")

pbar.close()

NameError: name 'channelsetup' is not defined

In [ ]:
# =============================================================================
# 1. CONFIG
# =============================================================================
#Extremely important, the number of candidates "n_half_circle" must be absolutely an odd number!!!!!!
class Config:
    def __init__(self,
                 R=1200.0,
                 amplitude_threshold=1/9,
                 dt=1/750000/6,
                 initial_pos=(0.0, 0.0),
                 domain_half_size=15000.0,
                 max_same_dir_visits=1,
                 max_steps=10,
                 threshold_init=0.12,
                 n_half_circle=7,
                 norm_factor=1/3,
                 branch_gradient_tol=0.5,
                 angular_probe_delta_deg=2,
                 rami_min_steps=3):
        self.R = R
        self.threshold = amplitude_threshold
        self.dt = dt
        self.initial_pos = np.array(initial_pos, dtype=float)
        self.domain = domain_half_size
        self.max_dir_visits = max_same_dir_visits
        self.max_steps = max_steps
        self.threshold_init  = threshold_init   # step 1: skip direction if norm < threshold_init
        self.n_half_circle   = n_half_circle    # number of probe points on the half-circle scan
        self.norm_factor     = norm_factor      # U0 normalisation divisor: |U| / (norm_factor * U0)
        self.branch_gradient_tol     = branch_gradient_tol      # max relative drop (A1-A2)/A1 to allow a split
        self.angular_probe_delta_deg = angular_probe_delta_deg  # +/-angle (deg) probed around each candidate
        self.rami_min_steps          = rami_min_steps           # discard ramification if <= this many steps
cfg = Config()

In [ ]:
#To find the intial point
# Positions: colonnes x,y
pos = pd.read_csv(pos_file, header=None)
pos[[0, 1]] = pos[[0, 1]] * 1e6  # m → µm
# --- build KDTree ---
tree = cKDTree(pos[[0, 1]].values)
#We charge the whole file of displacement here
print("Upload the signals in memory... It will take some times (around 50mins)-- Go and have a lunch, don't forget to Smile, it a free theurapy")
disp_data=pd.read_csv(disp_file, header=None)
print("Done, see! You had time to breath, have a cofee and smile")

p0 = np.array([0.0, 0.0])
#find the maximal amplitude of the initial point
_, idx_pos = tree.query(p0)
## Signal associated to this point
signal = disp_data.iloc[:, idx_pos].values
idx_max = np.abs(signal).argmax()
# Amplitude maximale (en valeur absolue)
amp_max = np.abs(signal[idx_max])
U_norm = amp_max * cfg.norm_factor

U_initial=amp_max/U_norm

Upload the signals in memory... It will take some times (around 50mins)-- Go and have a lunch, don't forget to Smile, it a free theurapy


ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [103]:
#To determine the 4 first point 
p0 = np.array([0.0, 0.0])
main_angles = np.linspace(0, 2*np.pi, 4, endpoint=False)
cpts  = p0 + cfg.R * np.column_stack([np.cos(main_angles), np.sin(main_angles)])
# --- project onto real mesh ---
dist, idx_init = tree.query(cpts)

cpts_projected = pos.iloc[idx_init].values

print("Ideal points:\n", cpts)
print("Projected points:\n", cpts_projected)


Ideal points:
 [[ 2.40000000e+03  0.00000000e+00]
 [ 1.46957616e-13  2.40000000e+03]
 [-2.40000000e+03  2.93915232e-13]
 [-4.40872848e-13 -2.40000000e+03]]
Projected points:
 [[ 2400.     0.     0.]
 [    0.  2400.     0.]
 [-2400.     0.     0.]
 [    0. -2400.     0.]]


In [ ]:
# =============================================================================
# 2. MAIN MOTION AND EXPLORATION LOOP
# =============================================================================

coordinates = cpts_projected  # The 4 initial starting positions
log_file = os.path.join(os.path.dirname(pos_file), 'tracking_log.csv')

# Create the CSV file with the updated header
with open(log_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['branch_id', 'step', 'x_um', 'y_um', 'amp_norm', 'time_step_idx', 'fallback_used'])

# Log the initial point p0
real_x0, real_y0,real_z0 = pos.iloc[idx_pos].values
with open(log_file, 'a', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([-1, 0, real_x0, real_y0, U_initial, idx_max, False])

# Step limit to avoid tracking indefinitely on a single branch
max_steps_per_branch = cfg.max_steps  
tol_time = 3

# Origin position
p0_projected=pos.iloc[idx_pos].values
p0_x, p0_y = p0_projected[0], p0_projected[1]
#Initial position
initial_x, initial_y=p0_projected[0], p0_projected[1]

for i in range(len(coordinates)):
    print(f"\n{'='*40}\nSTARTING BRANCH {i+1}/{len(coordinates)}\n{'='*40}")
    
    # Start at the center to initiate the trajectory towards cpts_projected[i]
    current_x, current_y = p0_x, p0_y
    target_x = coordinates[i][0]
    target_y = coordinates[i][1]
    
    # Initial branch orientation
    cur_angle = np.arctan2(target_y - current_y, target_x - current_x)
    
    # ---------------------------------------------------------
    # Step A: Move to the starting point of the branch
    # ---------------------------------------------------------
    delta_x = round(target_x - current_x)
    delta_y = round(target_y - current_y)
    xdir = np.sign(delta_x) if delta_x != 0 else 1
    ydir = np.sign(delta_y) if delta_y != 0 else 1
    stepx, stepy = abs(delta_x), abs(delta_y)

    if stepx > 0:
        stage_motion_x(step_size_x=stepx, direction_x=xdir)
        while scan_stage.is_moving(): time.sleep(0.1)
    if stepy > 0:
        stage_motion_y(step_size_y=stepy, direction_y=ydir)
        while scan_stage.is_moving(): time.sleep(0.1)

    # Measure the initial signal (t_old)
    real_x, real_y, amp_norm, t_old = measure_signal_at_point(target_x, target_y, tree, disp_data, pos, U_norm)
    current_x, current_y = real_x, real_y
    
    # Log the branch initialization
    with open(log_file, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([i, 0, current_x, current_y, amp_norm, t_old, False])
        
    print(f"Branch {i} - Origin reached: ({current_x:.1f}, {current_y:.1f}) | amp={amp_norm:.4f} | t_old={t_old}")

    # ---------------------------------------------------------
    # Step B: Local "Branch First" Search (Semi-circle)
    # ---------------------------------------------------------
    for step in range(1, max_steps_per_branch + 1):
        
        # Save the origin of this step for angle calculation
        origin_x, origin_y = current_x, current_y
        
        # 1. Generate candidate points in a semi-circle in the direction of propagation
        cand_pts = generate_semi_circle_points(current_x, current_y, cur_angle, cfg.R, nh=cfg.n_half_circle,ang_probe_del=cfg.angular_probe_delta_deg)
        _, idx_pjted = tree.query(cand_pts)
        cand_pts_projected=pos.iloc[idx_pjted].values
        # 2. Measure amplitudes and temporal indices via the KDTree (WITH PHYSICAL MOTION)
        candidates = []
        for pt_x, pt_y,_ in cand_pts_projected:
            
            # ---> PHYSICAL MOTION TO CANDIDATE POINT <---
            delta_x = round(pt_x - current_x)
            delta_y = round(pt_y - current_y)
            xdir = np.sign(delta_x) if delta_x != 0 else 1
            ydir = np.sign(delta_y) if delta_y != 0 else 1
            stepx, stepy = abs(delta_x), abs(delta_y)

            if stepx > 0:
                stage_motion_x(step_size_x=stepx, direction_x=xdir)
                while scan_stage.is_moving(): time.sleep(0.1)
            if stepy > 0:
                stage_motion_y(step_size_y=stepy, direction_y=ydir)
                while scan_stage.is_moving(): time.sleep(0.1)
                
            time.sleep(2.5)
            
            # Measure
            cand_data = measure_signal_at_point(pt_x, pt_y, tree, disp_data, pos, U_norm)
            candidates.append(cand_data)
            
            # Update physical position tracking to the real projected coordinates
            current_x, current_y = cand_data[0], cand_data[1] 
            
        # 3. Select the best candidate with the temporal constraint
        best_cand, fallback = select_best_candidate(candidates, t_old, tol=tol_time)
        next_x, next_y, next_amp, next_t = best_cand
        
        # Stopping criterion
        if next_amp < cfg.threshold:
            print(f"Branch {i} terminated: signal too weak ({next_amp:.4f}).")
            break
        
        # 4. Update the angle from step origin to best candidate
        #Note: Angle is calculated relative to the base of the arc, not the last visited candidate!
        #best_cand[0] and [1] are the coordinates of the chosen point. We must calculate the angle
        #from the origin of this step (which was the best point of the previous step).
        #Howeever, current _x and current_y have moved! Let's calculate the angle based on where we are going
        nhc=cfg.n_half_circle
        nhcp=int((3*nhc-1)/2)
        print(nhcp)
      
        cur_angle = np.arctan2(next_y - cand_pts_projected[nhcp][1], next_x - cand_pts_projected[nhcp][0])
        
        # 5. Physical stage movement BACK to the best candidate
        delta_x = round(next_x - current_x)
        delta_y = round(next_y - current_y)
        xdir = np.sign(delta_x) if delta_x != 0 else 1
        ydir = np.sign(delta_y) if delta_y != 0 else 1
        stepx, stepy = abs(delta_x), abs(delta_y)

        if stepx > 0:
            stage_motion_x(step_size_x=stepx, direction_x=xdir)
            while scan_stage.is_moving(): time.sleep(0.1)
        if stepy > 0:
            stage_motion_y(step_size_y=stepy, direction_y=ydir)
            while scan_stage.is_moving(): time.sleep(0.1)
        
        time.sleep(2.5)
            
        # 6. Update current state and Log
        current_x, current_y = next_x, next_y
        t_old = next_t
        
        with open(log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([i, step, current_x, current_y, next_amp, next_t, fallback])
            
        print(f"   -> Step {step}: ({current_x:.1f}, {current_y:.1f}) | amp={next_amp:.4f} | t={next_t} | fallback={fallback}")

    # ---------------------------------------------------------
    # Step C: Return to the center before exploring the next branch
    # ---------------------------------------------------------
    print(f"Branch {i} complete. Returning to center p0...")
    delta_x = round(p0_x - current_x)
    delta_y = round(p0_y - current_y)
    
    if abs(delta_x) > 0:
        stage_motion_x(step_size_x=abs(delta_x), direction_x=np.sign(delta_x))
        while scan_stage.is_moving(): time.sleep(0.1)
    if abs(delta_y) > 0:
        stage_motion_y(step_size_y=abs(delta_y), direction_y=np.sign(delta_y))
        while scan_stage.is_moving(): time.sleep(0.1)

print(f"\nComplete tracking. Logs saved to {log_file}")

# ---------------------------------------------------------
# Step D: Return to the center 
# ---------------------------------------------------------
print(" Returning to the initial position..")
delta_x = round(p0_x - current_x)
delta_y = round(p0_y - current_y)

if abs(delta_x) > 0:
    stage_motion_x(step_size_x=abs(delta_x), direction_x=np.sign(delta_x))
    while scan_stage.is_moving(): time.sleep(0.1)
if abs(delta_y) > 0:
    stage_motion_y(step_size_y=abs(delta_y), direction_y=np.sign(delta_y))
    while scan_stage.is_moving(): time.sleep(0.1)



STARTING BRANCH 1/4
Mon Jun 22 16:06:21 2026   Moving stage in X- by 2400 microns...
Mon Jun 22 16:06:21 2026   Current position (-15592)
Branch 0 - Origin reached: (2400.0, 0.0) | amp=0.2959 | t_old=43
Mon Jun 22 16:06:29 2026   Moving stage in Y- by 2400 microns...
Mon Jun 22 16:06:29 2026   Current position (-607036)
Mon Jun 22 16:06:32 2026   Moving stage in X- by 1700 microns...
Mon Jun 22 16:06:32 2026   Current position (967448)
Mon Jun 22 16:06:38 2026   Moving stage in Y- by 700 microns...
Mon Jun 22 16:06:38 2026   Current position (-1590059)
Mon Jun 22 16:06:40 2026   Moving stage in X- by 700 microns...
Mon Jun 22 16:06:40 2026   Current position (1663769)
Mon Jun 22 16:06:43 2026   Moving stage in Y- by 1700 microns...
Mon Jun 22 16:06:43 2026   Current position (-1303338)
Mon Jun 22 16:06:46 2026   Moving stage in X- by 700 microns...
Mon Jun 22 16:06:46 2026   Current position (1950468)
Mon Jun 22 16:06:48 2026   Moving stage in Y- by 1700 microns...
Mon Jun 22 16:06:48

In [110]:
# ---------------------------------------------------------
# Step D: Return to the center 
# ---------------------------------------------------------
print(" Returning to the initial position..")
delta_x = round(p0_x - current_x)
delta_y = round(p0_y - current_y)

if abs(delta_x) > 0:
    stage_motion_x(step_size_x=abs(delta_x), direction_x=np.sign(delta_x))
    while scan_stage.is_moving(): time.sleep(0.1)
if abs(delta_y) > 0:
    stage_motion_y(step_size_y=abs(delta_y), direction_y=np.sign(delta_y))
    while scan_stage.is_moving(): time.sleep(0.1)

 Returning to the initial position..
Mon Jun 22 16:03:15 2026   Moving stage in X- by 2400 microns...
Mon Jun 22 16:03:15 2026   Current position (4005884)


In [ ]:

# -----------------------------------------------------
# 1. HIGH-RESOLUTION SCAN PARAMETERS
# -----------------------------------------------------
log_file = r'C:\Users\tvreugdenhil\Desktop\juan\Figure8data\tracking_log_2400.csv' # Our tracking file
buffer_radius_um = 1000.0           # +/- 1 mm around the segments
resolution_um = 100.0               # Spatial step (measurement density) (e.g., 100 µm)

# -----------------------------------------------------
# 2. READING AND GEOMETRY CREATION
# -----------------------------------------------------
df = pd.read_csv(log_file)

# We will store all the propagation lines
lines = []
for branch_id, group in df.groupby('branch_id'):
    # Ignore the origin line if it's isolated; we want the segments
    if len(group) > 1:
        coords = group[['x_um', 'y_um']].values
        lines.append(LineString(coords))

# Create a global geometry combining all segments
# and apply the "buffer" to create the +/- 1 mm zone
if len(lines) > 1:
    from shapely.ops import unary_union
    full_trajectory = unary_union(lines)
else:
    full_trajectory = lines[0]

scan_zone = full_trajectory.buffer(buffer_radius_um)

# -----------------------------------------------------
# 3. GRID GENERATION, FILTERING AND PROJECTION
# -----------------------------------------------------
# Define the total bounding box
minx, miny, maxx, maxy = scan_zone.bounds

x_vals = np.arange(minx, maxx + resolution_um, resolution_um)
y_vals = np.arange(miny, maxy + resolution_um, resolution_um)

X, Y = np.meshgrid(x_vals, y_vals)
grid_points = np.column_stack([X.ravel(), Y.ravel()])

# Keep only the points strictly inside the buffer zone
hr_points_unsorted = []
for pt in grid_points:
    if scan_zone.contains(Point(pt)):
        hr_points_unsorted.append(pt)

hr_points_unsorted = np.array(hr_points_unsorted)
print(f"Number of theoretical points in the zone: {len(hr_points_unsorted)}")

# --- PROJECTION ONTO THE PHYSICAL GRID (cKDTree) ---
# We use the 'tree' object and 'pos' dataframe already created at the beginning of your script
print("Projecting points onto the physical grid...")
distances, indices = tree.query(hr_points_unsorted)

# Retrieve the real coordinates of the mesh
hr_points_projected = pos.iloc[indices][[0, 1]].values

# Remove duplicates (multiple theoretical points might fall on the same physical point)
hr_points_unique = np.unique(hr_points_projected, axis=0)
print(f"Number of unique physical points to scan: {len(hr_points_unique)}")


# -----------------------------------------------------
# 4. PATH OPTIMIZATION (Nearest Neighbor) & FAST SAVING
# -----------------------------------------------------
# To prevent the stage from making erratic back-and-forth movements
current_point = np.array([0.0, 0.0]) # Initial stage position

unvisited = hr_points_unique.tolist() # We now use the projected and unique points
sorted_hr_points = []

print("Optimizing trajectory (Nearest Neighbor)...")
while unvisited:
    # Find the closest point to the current position
    dists = cdist([current_point], unvisited)[0]
    closest_idx = np.argmin(dists)
    
    current_point = unvisited.pop(closest_idx)
    sorted_hr_points.append(current_point)

sorted_hr_points = np.array(sorted_hr_points)

# --- ULTRA-FAST SAVING (.npy) ---
save_path_npy = r'C:\Users\tvreugdenhil\Desktop\juan\Figure8data\hr_scan_points.npy'
np.save(save_path_npy, sorted_hr_points)
print(f"Points quickly saved in binary format at: {save_path_npy}")
# To load them later if needed: loaded_points = np.load(save_path_npy)

# (Optional) Visualization
plot.figure(figsize=(8, 8))
x_ext, y_ext = scan_zone.exterior.xy
plot.plot(x_ext, y_ext, color='red', alpha=0.5, label='Tolerance zone (+/- 1mm)')
plot.scatter(df['x_um'], df['y_um'], c='black', zorder=5, label='Tracking points (Log)')
# Display the projected points
plot.scatter(sorted_hr_points[:, 0], sorted_hr_points[:, 1], c='blue', s=5, alpha=0.7, label='Projected HR points')
plot.title('High-Resolution Scan (Projected onto the physical grid)')
plot.xlabel('X (µm)')
plot.ylabel('Y (µm)')
plot.legend()
plot.axis('equal')
plot.show()

In [ ]:
#(SIMULATION MODE): Here we don't need the oscilloscope measurement
# -----------------------------------------------------
# AUTOMATED 2D HIGH-RES SCAN LOOP 
# -----------------------------------------------------
number_points = len(sorted_hr_points)

# We assume the stage is at the absolute origin (0,0) at the start of the script
current_x, current_y = initial_x, initial_y

# Setup progress bar
pbar = tqdm(total=number_points, ncols=100, unit="pt", dynamic_ncols=True, mininterval=0.5)

print("Starting simulation loop (Stage movement only, no scope acquisition)...")

for I in range(number_points):
    target_x, target_y = sorted_hr_points[I]
    timestamp = time.strftime('%H:%M:%S')

    # --- 1. PHYSICAL STAGE MOVEMENT ---
    desc = f"{timestamp} | Point #{I+1}/{number_points} | Status: Stage moving"
    pbar.set_description(desc)
    
    delta_x = target_x - current_x
    delta_y = target_y - current_y
    
    xdir = np.sign(delta_x) if delta_x != 0 else 1
    ydir = np.sign(delta_y) if delta_y != 0 else 1
    stepx, stepy = abs(delta_x), abs(delta_y)

    if stepx > 0:
        stage_motion_x(step_size_x=stepx, direction_x=xdir)
        while scan_stage.is_moving(): 
            time.sleep(0.1)
        
    if stepy > 0:
        stage_motion_y(step_size_y=stepy, direction_y=ydir)
        while scan_stage.is_moving(): 
            time.sleep(0.1)
        
    # Update current tracked coordinates after movement is done
    current_x = target_x
    current_y = target_y

    # --- 2. SIMULATED MEASUREMENT (3-SECOND PAUSE) ---
    desc = f"{timestamp} | Point #{I+1}/{number_points} | Status: Simulating measurement"
    pbar.set_description(desc)
    
    time.sleep(3.0)  # 3 seconds delay to fake the actual pulse/acquisition

    # --- 3. PROGRESS UPDATE ---
    status = "Position checked"
    desc = f"{timestamp} | Point #{I+1}/{number_points} | Status: {status}"
    pbar.set_description(desc)
    pbar.update(1)

# ----------------------------
# Homing and Return to origin
# ----------------------------
pbar.close()

print(f"\n{time.ctime()}   Homing stages to (0,0)...")
if current_x != 0:
    stage_motion_x(step_size_x=abs(current_x), direction_x=np.sign(-current_x))
    while scan_stage.is_moving(): 
        time.sleep(0.1)
        
if current_y != 0:
    stage_motion_y(step_size_y=abs(current_y), direction_y=np.sign(-current_y))
    while scan_stage.is_moving(): 
        time.sleep(0.1)
        
print(f"{time.ctime()}   Homing complete. Simulation successful.")

In [ ]:
# -----------------------------------------------------
# AUTOMATED 2D HIGH-RES SCAN LOOP
# -----------------------------------------------------
number_points = len(sorted_hr_points)

# We assume the stage is at the absolute origin (0,0) at the start of the script
current_x, current_y = initial_x, initial_y

# Setup progress bar
pbar = tqdm(total=number_points, ncols=100, unit="acq", dynamic_ncols=True, mininterval=0.5)

# Arm function generator
fg.arm_ch2_for_manual_pulses()

for I in range(number_points):
    target_x, target_y = sorted_hr_points[I]
    timestamp = time.strftime('%H:%M:%S')

    # --- 1. STAGE MOVEMENT ---
    desc = f"{timestamp} | Acq #{I+1}/{number_points} | Status: Stage moving"
    pbar.set_description(desc)
    
    delta_x = target_x - current_x
    delta_y = target_y - current_y
    
    xdir = np.sign(delta_x) if delta_x != 0 else 1
    ydir = np.sign(delta_y) if delta_y != 0 else 1
    stepx, stepy = abs(delta_x), abs(delta_y)

    if stepx > 0:
        stage_motion_x(step_size_x=stepx, direction_x=xdir)
        while scan_stage.is_moving(): time.sleep(0.1)
        
    if stepy > 0:
        stage_motion_y(step_size_y=stepy, direction_y=ydir)
        while scan_stage.is_moving(): time.sleep(0.1)
        
    # Short pause to minimize motor vibrations before acquisition
    time.sleep(1.0) 

    # Update current coordinates
    current_x = target_x
    current_y = target_y

    # --- 2. OSCILLOSCOPE CONFIGURATION & ACQUISITION ---
    data_file = f"\\test_HR_{I}.csv"
    data_full_path = data_path + data_file

    desc = f"{timestamp} | Acq #{I+1}/{number_points} | Status: Configuring scope"
    pbar.set_description(desc)
    
    channelsetup(scope, chsetup, n_active_channel)
    timebase(scope, acquisition_time, sampling_rate, pretrigger)
    trigger_setup(trig_mode=trig_mode, trig_polarity=trig_polarity, trig_source=trig_source, trig_level=trig_level)
    data_acquisition(acquisition[acq], num_average, None)

    while scope.query('trigger:state?').strip() != 'READY':
        time.sleep(0.1)

    fg.trigger_ch2_pulse()

    while int(scope.query('acquire:state?')) != 0:
        time.sleep(0.1)

    # --- 3. SAVING ---
    wfm_out, wfm_raw = save_data(save_chan=save_chn, data_full_path=data_full_path) 
    
    if os.path.isfile(data_full_path) and os.access(data_full_path, os.R_OK):
        status = "Data saved"
    else:
        status = "Error saving data"
        pbar.close()
        raise RuntimeError(f"{timestamp}   Error saving data for acquisition #{I+1}")

    desc = f"{timestamp} | Acq #{I+1}/{number_points} | Status: {status}"
    pbar.set_description(desc)
    pbar.update(1)

# ----------------------------
# Cleanup and Return to origin
# ----------------------------
fg.disarm_ch2()
scope.write('LOCK NONE')
pbar.close()

print(f"{time.ctime()}   Homing stages to (0,0)...")
if current_x != 0:
    stage_motion_x(step_size_x=abs(current_x), direction_x=np.sign(-current_x))
    while scan_stage.is_moving(): time.sleep(0.1)
if current_y != 0:
    stage_motion_y(step_size_y=abs(current_y), direction_y=np.sign(-current_y))
    while scan_stage.is_moving(): time.sleep(0.1)
print(f"{time.ctime()}   Homing complete.")

In [163]:
# stage motion manual X-direction
#scope.write('LOCK NONE')
stage_motion_x(step_size_x =10, direction_x = -1)

Mon Jun 22 16:26:32 2026   Moving stage in X- by 10 microns...
Mon Jun 22 16:26:32 2026   Current position (37640)


In [159]:
# stage motion manual Y-direction
#scope.write('LOCK NONE')
stage_motion_y(step_size_y = 10, direction_y = -1)

Mon Jun 22 16:26:12 2026   Moving stage in Y- by 10 microns...
Mon Jun 22 16:26:12 2026   Current position (-217099)


In [ ]:
# example code for moving in four coordinates

coordinates = np.array([[0,0],[100,0],[100,100],[0,100],[0,0]]) # positions in microns!!

# keep track of where the stage actually is
current_x = 0
current_y = 0

for i in range(len(coordinates)):
    target_x = coordinates[i][0]
    target_y = coordinates[i][1]
    
    # calculate relative distance and direction
    delta_x = target_x - current_x
    delta_y = target_y - current_y
    
    xdir = np.sign(delta_x) if delta_x != 0 else 1
    ydir = np.sign(delta_y) if delta_y != 0 else 1
    
    # absolute step sizes
    stepx = abs(delta_x)
    stepy = abs(delta_y)

    # only move X if there is a distance to cover
    if stepx > 0:
        stage_motion_x(step_size_x=stepx, direction_x=xdir)
        while scan_stage.is_moving():
            time.sleep(0.1) # wait until moving is done
        time.sleep(2.5) # wait for safety to minimize vibrations. We can tune the time it has to wait to be anything

    # only move Y if there is a distance to cover
    if stepy > 0:
        stage_motion_y(step_size_y=stepy, direction_y=ydir)
        while scan_stage.is_moving():
            time.sleep(0.1)
        time.sleep(2.5)

    ### FUNCTION TO GET DISPLACEMENT DATA ###
        
    # Update current position tracking
    current_x = target_x
    current_y = target_y

In [ ]:
scan_stage.close()

In [112]:
# Emergency/Manual stop !!! RUN ME TO STOP THE STAGES IMMEDIATELY !!!
scan_stage.stop(channel=1,immediate=True)
scan_stage.stop(channel=2,immediate=True)
scan_stage.get_status(channel=1)
scan_stage.get_status(channel=2)

['tracking', 'digio1', 'enabled']